In [ ]:
import os
import sys
import math
import shutil
from datetime import datetime, timezone, timedelta

# =====================================================================
# 1. ENVIRONMENT & PYSPARK WINDOWS INITIALIZATION
# =====================================================================
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

_hadoop_home = os.environ.get("HADOOP_HOME", "")
if _hadoop_home:
    os.environ["PATH"] = os.path.join(_hadoop_home, "bin") + ";" + os.environ.get("PATH", "")

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

print("[Gold Synthesizer] Initializing Smart Mispricing Engine v2 (Sentiment-Aware)...")
builder = SparkSession.builder \
    .appName("PredictIQ_Gold_Mispricing_Scores") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.minPartitionSize", "32mb") \
    .config("spark.sql.files.maxPartitionBytes", "128mb") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.parquet.enableVectorizedReader", "false")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Enable HTML table rendering natively inside Jupyter Notebooks
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)

ROOT = "C:\\Data Engineering\\codeprep\\predection_project"

kalshi_history_path = os.path.join(ROOT, "data", "silver", "kalshi_markets_history")
gdelt_summaries_path = os.path.join(ROOT, "data", "gold", "gdelt_summaries")
news_summaries_path = os.path.join(ROOT, "data", "gold", "news_summaries")
gold_path = os.path.join(ROOT, "data", "gold", "mispricing_scores")

[Gold Synthesizer] Initializing Smart Mispricing Engine v2 (Sentiment-Aware)...


In [ ]:

# =====================================================================
# 2. REFERENCE SETUP (REFERENCE ENGINE STRUCTURING)
# =====================================================================
ref_dir = os.path.join(ROOT, "data", "reference")
os.makedirs(ref_dir, exist_ok=True)
mapping_file = os.path.join(ref_dir, "kalshi_theme_map.csv")

if not os.path.exists(mapping_file):
    print("[Gold Synthesizer] kalshi_theme_map.csv not found. Creating default stub...")
    with open(mapping_file, "w") as f:
        f.write("series_ticker,gdelt_entity_type,gdelt_entity_name,required_regex\n")
        f.write("KXFED,organization,Federal Reserve,(Federal Reserve|FOMC|Jerome Powell|Fed Chair)\n")
        f.write("KXFED,theme,CENTRAL_BANK,(Federal Reserve|Federal Open Market|FOMC|Jerome Powell)\n")
        f.write("KXPOLITICS,theme,ELECTION,(U\\.S\\. Election|American Election|Senate Election|Midterm)\n")
        f.write("KXTW,location,Taiwan,(Taiwan|Taipei|ROC |TSMC|Cross-Strait)\n")
    print(f"[Gold Synthesizer] Default stub written to {mapping_file}")
else:
    print(f"[Gold Synthesizer] Using existing kalshi_theme_map.csv at {mapping_file}")

# =====================================================================
# 3. KALSHI HISTORY DATA EXTRACTION & DATA CLEANING
# =====================================================================
lookback_cutoff_dt  = datetime.now(timezone.utc) - timedelta(hours=48)
lookback_cutoff_str = lookback_cutoff_dt.strftime("%Y-%m-%d %H:%M:%S")
df_k = spark.read.format("delta").load(kalshi_history_path)

df_k = df_k.filter(
    F.col("ticker").isNotNull() &
    F.col("ingested_at").isNotNull() &
    (F.col("ingested_at").cast("timestamp") >= F.to_timestamp(F.lit(lookback_cutoff_str)))
)
print(f"[Gold Synthesizer] 48h window filter applied. Cutoff: {lookback_cutoff_str}")


In [ ]:
df_k.limit(50).toPandas()